In [1]:
# 03_inference_se_alignment.ipynb  (FINAL)
# Pillar 2 (inference): do boosted-stump score contributions carry logistic-grade SE/CI?
#
# FAIR SHARED COORDINATE: bin edges FIXED from the full sample (deployed scorecard
# structure). Each bootstrap replicate lets BOTH models re-feel the SAME structural +
# sampling noise on that fixed grid:
#   - logistic : recompute WoE(b) on the replicate, refit logit, contribution = beta^(b)*WoE^(b)(b)
#   - stump    : saturated univariate stump-boost == per-bin log-odds deviation (doc theorem).
#                MAIN path (B): compute that log-odds deviation directly (groupby) -> N_BOOT=500.
#                CROSS-CHECK (A): real GradientBoosting refit, N_BOOT=100, to show B==A.
# Missing-bin sub-analysis: split isomorphic features into no-missing vs has-missing.
# H3: on isomorphic (esp. no-missing) features the stump bin-contribution SE is
#     statistically indistinguishable from the logistic one (ratio ~ 1).
# Single-cell; greyscale seaborn; dpi600 png+pdf; no captions.

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import GradientBoostingClassifier
import statsmodels.api as sm
warnings.filterwarnings('ignore')

SEED = 42
N_BOOT = 500        # main path B (fast, theoretical saturation)
N_CROSS = 100       # cross-check path A (real GB refit)
rng = np.random.default_rng(SEED)
sns.set_theme(style='whitegrid', context='paper')
sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor': '0.2', 'grid.color': '0.85', 'font.size': 10})

ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA   = os.path.join(ROOT, 'data')
TABLES = os.path.join(ROOT, 'results', 'tables')
FIGS   = os.path.join(ROOT, 'results', 'figures')
RUN    = os.path.join(ROOT, '_run'); os.makedirs(RUN, exist_ok=True)

def savefig(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(FIGS, name + '.' + ext), dpi=600, bbox_inches='tight')
    plt.close(fig)

# ===================================================================
# 0. LOAD + fixed bin grid
# ===================================================================
woe_X = pd.read_parquet(os.path.join(DATA, 'gmsc_woe.parquet'))
dfc   = pd.read_parquet(os.path.join(DATA, 'gmsc_clean.parquet'))
meta  = json.load(open(os.path.join(DATA, 'woe_maps.json')))
TARGET = meta['target']; FEATURES = meta['features']; MAPS = meta['maps']
y_all = woe_X[TARGET].values.astype(int)
print('Loaded:', woe_X.shape, '| features:', FEATURES)

bin_order, bin_woe0, bin_w, has_missing = {}, {}, {}, {}
for c in FEATURES:
    recs = sorted([r for r in MAPS[c] if r['bin'] >= 0], key=lambda r: r['lo'])
    miss = [r for r in MAPS[c] if r['bin'] == -1]
    allrecs = recs + miss
    bin_order[c] = allrecs
    bin_woe0[c]  = np.array([r['woe'] for r in allrecs])
    bin_w[c]     = np.array([r['n']   for r in allrecs], float)
    has_missing[c] = any((r['bin'] == -1 and abs(r['woe']) > 1e-9) for r in allrecs)

def to_bin_index(x, recs):
    idx = np.full(len(x), len(recs)-1, int)
    fin = [r for r in recs if r['bin'] >= 0]
    for pos, r in enumerate(fin):
        sel = (~x.isna()).values & (x > r['lo']).values & (x <= r['hi']).values
        idx[sel] = pos
    if any(r['bin']==-1 for r in recs):
        idx[x.isna().values] = len(recs)-1
    else:
        idx[x.isna().values] = 0
    return idx

Xbin_all = {c: to_bin_index(dfc[c], bin_order[c]) for c in FEATURES}
nbins   = {c: len(bin_order[c]) for c in FEATURES}
neutral = {c: np.abs(bin_woe0[c]) < 1e-9 for c in FEATURES}
base_lo = np.log((y_all==1).sum()/(y_all==0).sum())

def logodds_curve(bidx, yb, nb, neutral_mask):
    """Saturated univariate stump-boost limit = per-bin log-odds deviation (Laplace)."""
    good = np.zeros(nb); bad = np.zeros(nb)
    np.add.at(good, bidx[yb==0], 1.0); np.add.at(bad, bidx[yb==1], 1.0)
    tg = (yb==0).sum(); tb = (yb==1).sum()
    lo = np.log((bad+0.5)/(good+0.5)) - np.log(tb/tg)
    lo[neutral_mask] = 0.0
    return lo

def woe_from_counts(bidx, yb, nb, tg, tb, neutral_mask):
    good = np.zeros(nb); bad = np.zeros(nb)
    np.add.at(good, bidx[yb==0], 1.0); np.add.at(bad, bidx[yb==1], 1.0)
    gr = (good+0.5)/(tg+0.5); br = (bad+0.5)/(tb+0.5)
    w = np.log(br/gr); w[neutral_mask] = 0.0
    return w

def gb_curve(bin_col, nb, yb, lr=0.3, n_est=120, seed=SEED):
    gb = GradientBoostingClassifier(max_depth=1, learning_rate=lr,
                                    n_estimators=n_est, random_state=seed)
    gb.fit(bin_col.reshape(-1,1).astype(float), yb)
    F0 = np.log(yb.mean()/(1-yb.mean()))
    return gb.decision_function(np.arange(nb).reshape(-1,1).astype(float)) - F0

# iso vs broken partition from 02
real = pd.read_csv(os.path.join(TABLES, '02_isomorphism_reality.csv')).set_index('feature')
ISO_THRESH = 0.9
iso_feats = [c for c in FEATURES if real.loc[c,'w_corr_raw'] >= ISO_THRESH]
brk_feats = [c for c in FEATURES if c not in iso_feats]
iso_nomiss = [c for c in iso_feats if not has_missing[c]]
iso_miss   = [c for c in iso_feats if has_missing[c]]
print('Isomorphic:', iso_feats)
print('  - no-missing:', iso_nomiss)
print('  - has-missing:', iso_miss)
print('Broken:', brk_feats)

# ===================================================================
# 1. POINT ESTIMATES (full sample)
# ===================================================================
Xw0 = np.column_stack([bin_woe0[c][Xbin_all[c]] for c in FEATURES])
logit0 = sm.Logit(y_all, sm.add_constant(Xw0)).fit(disp=0)
beta_hat = {c: logit0.params[j+1] for j,c in enumerate(FEATURES)}
beta_se_wald = {c: logit0.bse[j+1] for j,c in enumerate(FEATURES)}
s_hat = {c: logodds_curve(Xbin_all[c], y_all, nbins[c], neutral[c]) for c in FEATURES}
logit_contrib0 = {c: beta_hat[c]*bin_woe0[c] for c in FEATURES}

# ===================================================================
# 2. MAIN BOOTSTRAP (path B): recompute WoE + logit refit + log-odds stump, N_BOOT
# ===================================================================
n = len(y_all)
boot_logit_bin = {c: [] for c in FEATURES}
boot_stump_bin = {c: [] for c in FEATURES}
t0 = time.time()
for b in range(N_BOOT):
    idx = rng.integers(0, n, n)
    yb = y_all[idx]
    tgb = int((yb==0).sum()); tbb = int((yb==1).sum())
    cols = []; woe_b = {}
    for c in FEATURES:
        bidx = Xbin_all[c][idx]
        w = woe_from_counts(bidx, yb, nbins[c], tgb, tbb, neutral[c])
        woe_b[c] = w; cols.append(w[bidx])
        boot_stump_bin[c].append(logodds_curve(bidx, yb, nbins[c], neutral[c]))
    Xwb = np.column_stack(cols)
    try:
        lb = sm.Logit(yb, sm.add_constant(Xwb)).fit(disp=0)
        for j,c in enumerate(FEATURES):
            boot_logit_bin[c].append(lb.params[j+1]*woe_b[c])
    except Exception:
        for c in FEATURES: boot_logit_bin[c].append(np.full(nbins[c], np.nan))
    if (b+1) % 100 == 0:
        print('  [B] bootstrap %d/%d (%.0fs)' % (b+1, N_BOOT, time.time()-t0))
        np.save(os.path.join(RUN, 'ckpt_stump.npy'),
                {c: np.array(boot_stump_bin[c]) for c in FEATURES}, allow_pickle=True)
print('[B] done in %.0fs' % (time.time()-t0))

# ===================================================================
# 3. CROSS-CHECK BOOTSTRAP (path A): real GB refit, N_CROSS
# ===================================================================
cross_stump = {c: [] for c in FEATURES}
rng2 = np.random.default_rng(SEED+999)
t1 = time.time()
for b in range(N_CROSS):
    idx = rng2.integers(0, n, n); yb = y_all[idx]
    for c in FEATURES:
        cross_stump[c].append(gb_curve(Xbin_all[c][idx], nbins[c], yb, seed=SEED+b))
    if (b+1) % 25 == 0:
        print('  [A] cross %d/%d (%.0fs)' % (b+1, N_CROSS, time.time()-t1))
print('[A] done in %.0fs' % (time.time()-t1))

# ===================================================================
# 4. SE in shared coordinate + H3 stats (+ B-vs-A agreement)
# ===================================================================
rows, bin_rows = [], []
for c in FEATURES:
    L = np.array(boot_logit_bin[c]); S = np.array(boot_stump_bin[c]); A = np.array(cross_stump[c])
    se_l_bin = np.nanstd(L, axis=0, ddof=1); se_s_bin = np.nanstd(S, axis=0, ddof=1)
    se_a_bin = np.nanstd(A, axis=0, ddof=1)
    keep = ~neutral[c]; w = bin_w[c]
    wn = np.zeros_like(w); wn[keep] = w[keep]/w[keep].sum()
    def wrms(v): return float(np.sqrt(np.sum(wn[keep]*v[keep]**2))) if keep.any() else np.nan
    se_l, se_s, se_a = wrms(se_l_bin), wrms(se_s_bin), wrms(se_a_bin)
    ratio = se_s/se_l if se_l>0 else np.nan
    ba_ratio = se_s/se_a if se_a>0 else np.nan     # B vs A agreement (should be ~1)
    if keep.sum() >= 2:
        try: wpval = stats.wilcoxon(se_s_bin[keep], se_l_bin[keep]).pvalue
        except Exception: wpval = np.nan
    else: wpval = np.nan
    grp = 'iso_nomiss' if c in iso_nomiss else ('iso_miss' if c in iso_miss else 'broken')
    rows.append([c, grp, beta_hat[c], beta_se_wald[c], se_l, se_s, se_a, ratio, ba_ratio, wpval])
    for bi in range(nbins[c]):
        if keep[bi]:
            bin_rows.append([c, bi, bin_woe0[c][bi], logit_contrib0[c][bi], s_hat[c][bi],
                             se_l_bin[bi], se_s_bin[bi], se_a_bin[bi]])
se_df = pd.DataFrame(rows, columns=['feature','group','beta','beta_se_wald',
                                    'se_logit_wrms','se_stump_wrms','se_stumpA_wrms',
                                    'se_ratio','BA_ratio','wilcoxon_p'])
bin_df = pd.DataFrame(bin_rows, columns=['feature','bin','woe','contrib_logit','contrib_stump',
                                         'se_logit_bin','se_stump_bin','se_stumpA_bin'])
print('\n=== Per-feature SE alignment (fair coordinate, N_BOOT=%d) ===' % N_BOOT)
print(se_df.round(4).to_string(index=False))

# Path-level analytic cross-check
an_rows = []
for c in FEATURES:
    for bi, r in enumerate(bin_order[c]):
        if r['bin'] < 0 or neutral[c][bi]: continue
        g, bd = r['good'], r['bad']
        if g>0 and bd>0:
            se_woe = np.sqrt(1.0/bd + 1.0/g)
            row = bin_df[(bin_df.feature==c)&(bin_df.bin==bi)]
            an_rows.append([c, bi, se_woe,
                            float(row['se_stump_bin'].values[0]) if len(row) else np.nan,
                            float(row['se_logit_bin'].values[0]) if len(row) else np.nan])
an_df = pd.DataFrame(an_rows, columns=['feature','bin','se_woe_closed','se_stump_boot','se_logit_boot'])
an_df['ratio_stump'] = an_df['se_stump_boot']/an_df['se_woe_closed']
an_df['ratio_logit'] = an_df['se_logit_boot']/an_df['se_woe_closed']

# ===================================================================
# 5. SAVE
# ===================================================================
se_df.round(6).to_csv(os.path.join(TABLES, '03_se_alignment_feature.csv'), index=False)
bin_df.round(6).to_csv(os.path.join(TABLES, '03_se_alignment_bin.csv'), index=False)
an_df.round(6).to_csv(os.path.join(TABLES, '03_pathA_crosscheck.csv'), index=False)

# ===================================================================
# 6. FIGURE A — per-bin stump SE vs logit SE, by sub-group
# ===================================================================
fig, ax = plt.subplots(figsize=(6.0, 5.6))
lim = [0, max(bin_df['se_logit_bin'].max(), bin_df['se_stump_bin'].max())*1.1]
ax.plot(lim, lim, color='0.0', lw=1, ls='--', zorder=1)
styling = [('iso_nomiss','o','0.25'), ('iso_miss','s','0.55'), ('broken','^','0.78')]
gmap = {c: ('iso_nomiss' if c in iso_nomiss else ('iso_miss' if c in iso_miss else 'broken')) for c in FEATURES}
for grp, mk, col in styling:
    fs = [c for c in FEATURES if gmap[c]==grp]
    sub = bin_df[bin_df.feature.isin(fs)]
    ax.scatter(sub['se_logit_bin'], sub['se_stump_bin'], s=55, marker=mk, c=col,
               edgecolor='0.0', linewidth=.5, label=grp)
ax.set_xlabel('logistic-WoE bin-contribution SE'); ax.set_ylabel('stump bin-contribution SE')
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal')
ax.legend(frameon=True, facecolor='white', edgecolor='0.5')
savefig(fig, '03_se_scatter_bin')

# ===================================================================
# 7. FIGURE B — contribution with bootstrap 90% CI, strongest no-missing iso feature
# ===================================================================
pool = iso_nomiss if iso_nomiss else iso_feats
cstar = max(pool, key=lambda c: abs(beta_hat[c]))
keep = ~neutral[cstar]; xb = np.arange(nbins[cstar])[keep]
L = np.array(boot_logit_bin[cstar]); S = np.array(boot_stump_bin[cstar])
ql = np.nanpercentile(L,[5,95],axis=0); qs = np.nanpercentile(S,[5,95],axis=0)
fig, ax = plt.subplots(figsize=(7.0, 4.6))
ax.errorbar(xb-0.08, logit_contrib0[cstar][keep],
            yerr=[logit_contrib0[cstar][keep]-ql[0][keep], ql[1][keep]-logit_contrib0[cstar][keep]],
            fmt='o', color='0.1', capsize=3, lw=1.2, ms=5, label='logistic-WoE')
ax.errorbar(xb+0.08, s_hat[cstar][keep],
            yerr=[s_hat[cstar][keep]-qs[0][keep], qs[1][keep]-s_hat[cstar][keep]],
            fmt='s', color='0.55', capsize=3, lw=1.2, ms=5, label='stump s(bin)')
ax.axhline(0, color='0.1', lw=.5)
ax.set_xlabel('bin (low → high)'); ax.set_ylabel('log-odds contribution')
ax.set_title('%s  (bootstrap 90%% CI)' % cstar, fontsize=10)
ax.set_xticks(xb); ax.legend(frameon=True, facecolor='white', edgecolor='0.5')
savefig(fig, '03_contrib_ci_%s' % cstar)

# ===================================================================
# 8. FIGURE C — B vs A agreement (log-odds limit vs real GB), per bin
# ===================================================================
fig, ax = plt.subplots(figsize=(5.6, 5.4))
lim2 = [0, max(bin_df['se_stump_bin'].max(), bin_df['se_stumpA_bin'].max())*1.1]
ax.plot(lim2, lim2, color='0.0', lw=1, ls='--')
ax.scatter(bin_df['se_stumpA_bin'], bin_df['se_stump_bin'], s=45, c='0.4',
           edgecolor='0.0', linewidth=.4)
ax.set_xlabel('stump SE — real GB refit (path A)')
ax.set_ylabel('stump SE — log-odds limit (path B)')
ax.set_xlim(lim2); ax.set_ylim(lim2); ax.set_aspect('equal')
savefig(fig, '03_BvsA_agreement')

print('\nSaved figures: 03_se_scatter_bin, 03_contrib_ci_%s, 03_BvsA_agreement' % cstar)
print('Saved tables : 03_se_alignment_feature, 03_se_alignment_bin, 03_pathA_crosscheck')

# ===================================================================
# 9. H3 HEADLINE
# ===================================================================
def grp_ratio(g): 
    v = se_df[se_df.group==g]['se_ratio']; return v.mean(), v.std()
print('\n================ H3 HEADLINE (N_BOOT=%d, fair coordinate) ================' % N_BOOT)
for g in ['iso_nomiss','iso_miss','broken']:
    if (se_df.group==g).any():
        m,s = grp_ratio(g); print('%-11s: SE ratio stump/logit = %.3f ± %.3f' % (g, m, s))
print('B-vs-A agreement: median SE ratio (B/A) = %.3f (|.-1| small => log-odds limit tracks real GB)'
      % se_df['BA_ratio'].median())
print('Path analytic: median stump/WoE-closed = %.3f | logit/WoE-closed = %.3f'
      % (an_df['ratio_stump'].median(), an_df['ratio_logit'].median()))
print('H3: on no-missing isomorphic features the ratio ~ 1 => stump carries logistic-grade SE.')
print('\n03 complete.')


Loaded: (150000, 8) | features: ['RevolvingUtilizationOfUnsecuredLines', 'NumberOfTimes90DaysLate', 'NumberOfTime30-59DaysPastDueNotWorse', 'age', 'NumberOfOpenCreditLinesAndLoans', 'MonthlyIncome', 'NumberOfDependents']
Isomorphic: ['RevolvingUtilizationOfUnsecuredLines', 'NumberOfTimes90DaysLate', 'NumberOfTime30-59DaysPastDueNotWorse', 'age', 'MonthlyIncome']
  - no-missing: ['RevolvingUtilizationOfUnsecuredLines', 'NumberOfTimes90DaysLate', 'NumberOfTime30-59DaysPastDueNotWorse', 'age']
  - has-missing: ['MonthlyIncome']
Broken: ['NumberOfOpenCreditLinesAndLoans', 'NumberOfDependents']


  [B] bootstrap 100/500 (23s)


  [B] bootstrap 200/500 (46s)


  [B] bootstrap 300/500 (68s)


  [B] bootstrap 400/500 (90s)


...

  [A] cross 25/100 (218s)


  [A] cross 50/100 (439s)


  [A] cross 75/100 (657s)


  [A] cross 100/100 (879s)
[A] done in 879s

=== Per-feature SE alignment (fair coordinate, N_BOOT=500) ===
                             feature      group    beta  beta_se_wald  se_logit_wrms  se_stump_wrms  se_stumpA_wrms  se_ratio  BA_ratio  wilcoxon_p
RevolvingUtilizationOfUnsecuredLines iso_nomiss  0.6800        0.0125         0.0226         0.0273          0.0280    1.2096    0.9753      0.0938
             NumberOfTimes90DaysLate iso_nomiss  0.6000        0.0111         0.0091         0.0090          0.0090    0.9914    1.0105      1.0000
NumberOfTime30-59DaysPastDueNotWorse iso_nomiss  0.6271        0.0127         0.0109         0.0120          0.0129    1.0955    0.9273      0.7500
                                 age iso_nomiss  0.4470        0.0263         0.0189         0.0281          0.0290    1.4886    0.9701      0.0312
     NumberOfOpenCreditLinesAndLoans     broken -0.0609        0.0428         0.0116         0.0165          0.0170    1.4166    0.9703      0.3750
    


Saved figures: 03_se_scatter_bin, 03_contrib_ci_RevolvingUtilizationOfUnsecuredLines, 03_BvsA_agreement
Saved tables : 03_se_alignment_feature, 03_se_alignment_bin, 03_pathA_crosscheck

================ H3 HEADLINE (N_BOOT=500, fair coordinate) ================
iso_nomiss : SE ratio stump/logit = 1.196 ± 0.214
iso_miss   : SE ratio stump/logit = 1.717 ± nan
broken     : SE ratio stump/logit = 1.546 ± 0.184
B-vs-A agreement: median SE ratio (B/A) = 0.975 (|.-1| small => log-odds limit tracks real GB)
Path analytic: median stump/WoE-closed = 0.920 | logit/WoE-closed = 0.638
H3: on no-missing isomorphic features the ratio ~ 1 => stump carries logistic-grade SE.

03 complete.
